# Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import gc
import os
import sys
import time
from pathlib import Path

import psutil
import torch
from hydra.utils import instantiate
from omegaconf import OmegaConf
from torch.optim import AdamW

from src.utils.logger import setup_logging


setup_logging()

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.utils.notebook_setup import init_nlp_notebook #noqa E402


cfg = init_nlp_notebook()

if "paths" not in cfg:
    cfg.paths = OmegaConf.create()
cfg.paths.data_dir = str(PROJECT_ROOT / "data")

device = "cuda" if torch.cuda.is_available() else "cpu"

NLP Environment ready. Root: c:\nlp_template_decoder


# Data Preparation (Single Batch)

In [2]:
from src.core.data.builder import NLPDataModule


tokenizer = instantiate(cfg.model.tokenizer).build()
# Для обучения паддинг остается справа
tokenizer.padding_side = "right"

datamodule = NLPDataModule(data_cfg=cfg.data, tokenizer=tokenizer)
datamodule.prepare_data()
datamodule.setup(stage="fit")

train_dataloader = datamodule.train_dataloader()
train_iter = iter(train_dataloader)
batch_train = next(train_iter)

print("Batch keys:", batch_train.keys())
print("Input shape:", batch_train["input_ids"].shape)

c:\nlp_template_decoder\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:src.core.models.tokenization:Загрузка токенизатора: HuggingFaceM4/tiny-random-LlamaForCausalLM
[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got -1. This may result in unexpected behavior.
INFO:src.core.data.builder:Нашли кэш обработанных данных: c:\nlp_template_decoder\data/processed\sft_dataset_processed_b79be0ab. Подготовка пропущена.
c:\nlp_template_decoder\.venv\lib\site-packages\transformers\tokenization_utils_base.py:2368: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


Batch keys: KeysView({'input_ids': tensor([[ 4134,  1598,   278,  ...,     0,     0,     0],
        [ 5293,   278,  2183,  ...,   653,  7014, 29889],
        [ 4007,   647,   263,  ...,     0,     0,     0],
        ...,
        [ 6991,  3034,   675,  ...,     0,     0,     0],
        [12027,  7420,   596,  ...,     0,     0,     0],
        [16760,   701,   411,  ...,     0,     0,     0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]]), 'labels': tensor([[ -100,  -100,  -100,  ...,  -100,  -100,  -100],
        [ -100,  -100,  -100,  ...,   653,  7014, 29889],
        [ -100,  -100,  -100,  ...,  -100,  -100,  -100],
        ...,
        [ -100,  -100,  -100,  ...,  -100,  -100,  -100],
        [ -100,  -100,  -100,  ...,  -100,  -100,  -100],
        [ -100,  -100,  -100,  ...,  -100,  -100,  -

# Overfitting Probe Utility

In [3]:
from tabulate import tabulate


def quick_train_probe(target_modules, r, alpha, batch, cfg, device, lr=2e-4, n_steps=50):
    # 1. Динамически обновляем конфиг Hydra для PEFT
    cfg.model.peft.target_modules = target_modules
    cfg.model.peft.r = r
    cfg.model.peft.lora_alpha = alpha

    # 2. Собираем модель через фабрику
    model_builder = instantiate(cfg.model.builder, tokenizer=tokenizer)
    model = model_builder.build()
    model.train()

    if device == "cuda":
        model.to(device)
        batch = {k: v.to(device) for k, v in batch.items()}
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()

    trainable_params, all_param = model.get_nb_trainable_parameters()
    trainable_pct = 100 * trainable_params / all_param

    opt = AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)

    start_time = time.time()
    initial_loss, final_loss = 0.0, 0.0

    for step in range(n_steps):
        opt.zero_grad()
        # CausalLMOutput возвращает loss, если переданы labels
        outputs = model(**batch)
        loss = outputs.loss

        if step == 0:
            initial_loss = loss.item()
        if step == n_steps - 1:
            final_loss = loss.item()

        loss.backward()
        opt.step()

    if device == "cuda":
        torch.cuda.synchronize()
        peak_mem_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
    else:
        peak_mem_mb = psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2)

    train_time = time.time() - start_time

    # Очистка памяти
    del model, outputs, loss, opt, model_builder
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

    return {
        "Target Modules": "+".join(target_modules),
        "r/alpha": f"{r}/{alpha}",
        "Params (%)": round(trainable_pct, 4),
        "Peak Mem (MB)": round(peak_mem_mb, 1),
        "Time (s)": round(train_time, 2),
        "Start Loss": round(initial_loss, 4),
        "End Loss": round(final_loss, 4)
    }

# Experiment 1: Target Modules

In [4]:
# LLM используют другие названия слоев по сравнению с BERT
configs_to_test = [
    ["q_proj", "v_proj"], # Минимальный стандарт
    ["q_proj", "k_proj", "v_proj", "o_proj"], # Все слои внимания
    ["q_proj", "v_proj", "gate_proj", "up_proj", "down_proj"], # Внимание + MLP
]

test_lr = 2e-4
test_steps = 50
r_fixed = 8
alpha_fixed = 16

print(f"Тестируем {len(configs_to_test)} конфигураций target_modules")
print(f"({test_steps} шагов, LR={test_lr}, r={r_fixed}, alpha={alpha_fixed})\n")

probe_results = []
for targets in configs_to_test:
    res = quick_train_probe(targets, r_fixed, alpha_fixed, batch_train, cfg, device, test_lr, test_steps)
    probe_results.append(res)
    print(f"[{res['Target Modules']}] | End Loss: {res['End Loss']:.4f}")

print("\n=== Сводная таблица (Target Modules) ===")
print(tabulate(probe_results, headers="keys", tablefmt="pipe", floatfmt=".4f"))

Тестируем 3 конфигураций target_modules
(50 шагов, LR=0.0002, r=8, alpha=16)



INFO:src.core.models.builder:Загрузка базовой архитектуры: HuggingFaceM4/tiny-random-LlamaForCausalLM
INFO:src.core.models.builder:Применение квантизации BitsAndBytes.
[transformers] The following generation flags are not valid and may be ignored: ['pad_token_id']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loading weights: 100%|██████████| 21/21 [00:00<00:00, 1166.92it/s]
INFO:src.core.models.builder:Активация Gradient Checkpointing (Экономия VRAM).
INFO:src.core.models.builder:Режим PEFT: Инициализация нового LoRA адаптера.
INFO:src.core.models.builder:LoRA: 1,024 обучаемых из 1,033,296 (0.0991%)
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
c:\nlp_template_decoder\.venv\lib\site-packages\torch\_dynamo\eval_frame.py:1446: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=

[q_proj+v_proj] | End Loss: 10.3756


Loading weights: 100%|██████████| 21/21 [00:00<00:00, 505.89it/s]
INFO:src.core.models.builder:Активация Gradient Checkpointing (Экономия VRAM).
INFO:src.core.models.builder:Режим PEFT: Инициализация нового LoRA адаптера.
INFO:src.core.models.builder:LoRA: 2,048 обучаемых из 1,034,320 (0.1980%)
INFO:src.core.models.builder:Загрузка базовой архитектуры: HuggingFaceM4/tiny-random-LlamaForCausalLM
INFO:src.core.models.builder:Применение квантизации BitsAndBytes.


[q_proj+k_proj+v_proj+o_proj] | End Loss: 10.3727


Loading weights: 100%|██████████| 21/21 [00:00<00:00, 583.36it/s]
INFO:src.core.models.builder:Активация Gradient Checkpointing (Экономия VRAM).
INFO:src.core.models.builder:Режим PEFT: Инициализация нового LoRA адаптера.
INFO:src.core.models.builder:LoRA: 4,864 обучаемых из 1,037,136 (0.4690%)


[q_proj+v_proj+gate_proj+up_proj+down_proj] | End Loss: 10.3717

=== Сводная таблица (Target Modules) ===
| Target Modules                            | r/alpha   |   Params (%) |   Peak Mem (MB) |   Time (s) |   Start Loss |   End Loss |
|:------------------------------------------|:----------|-------------:|----------------:|-----------:|-------------:|-----------:|
| q_proj+v_proj                             | 8/16      |       0.0991 |       1047.1000 |    28.0300 |      10.3786 |    10.3756 |
| q_proj+k_proj+v_proj+o_proj               | 8/16      |       0.1980 |       1342.7000 |    26.5200 |      10.3786 |    10.3727 |
| q_proj+v_proj+gate_proj+up_proj+down_proj | 8/16      |       0.4690 |       1343.5000 |    25.9300 |      10.3786 |    10.3717 |


# Experiment 2: LoRA Rank (r) and Alpha

In [5]:
# Берем лучшую конфигурацию из предыдущего эксперимента (например, первую)
best_targets = configs_to_test[1]

r_alpha_pairs = [
    (8, 8),   # ratio 1.0
    (8, 16),  # ratio 2.0 (standard)
    (16, 32), # ratio 2.0 (higher capacity)
    (32, 64)  # ratio 2.0 (max capacity)
]

print(f"Тестируем ранги для таргетов: {best_targets}\n")

ra_results = []
for r, alpha in r_alpha_pairs:
    res = quick_train_probe(best_targets, r, alpha, batch_train, cfg, device, test_lr, test_steps)
    ra_results.append(res)
    print(f"[r={r}, alpha={alpha}] | End Loss: {res['End Loss']:.4f} | Peak Mem: {res['Peak Mem (MB)']} MB")

print("\n=== Сводная таблица (Rank & Alpha) ===")
print(tabulate(ra_results, headers="keys", tablefmt="pipe", floatfmt=".4f"))

best_ra = min(ra_results, key=lambda x: x["End Loss"])
print(f"\nСамая мощная конфигурация: r/alpha = {best_ra['r/alpha']} с End Loss = {best_ra['End Loss']}")

INFO:src.core.models.builder:Загрузка базовой архитектуры: HuggingFaceM4/tiny-random-LlamaForCausalLM
INFO:src.core.models.builder:Применение квантизации BitsAndBytes.


Тестируем ранги для таргетов: ['q_proj', 'k_proj', 'v_proj', 'o_proj']



Loading weights: 100%|██████████| 21/21 [00:00<00:00, 2247.92it/s]
INFO:src.core.models.builder:Активация Gradient Checkpointing (Экономия VRAM).
INFO:src.core.models.builder:Режим PEFT: Инициализация нового LoRA адаптера.
INFO:src.core.models.builder:LoRA: 2,048 обучаемых из 1,034,320 (0.1980%)
INFO:src.core.models.builder:Загрузка базовой архитектуры: HuggingFaceM4/tiny-random-LlamaForCausalLM
INFO:src.core.models.builder:Применение квантизации BitsAndBytes.


[r=8, alpha=8] | End Loss: 10.3758 | Peak Mem: 1098.9 MB


Loading weights: 100%|██████████| 21/21 [00:00<00:00, 1909.77it/s]
INFO:src.core.models.builder:Активация Gradient Checkpointing (Экономия VRAM).
INFO:src.core.models.builder:Режим PEFT: Инициализация нового LoRA адаптера.
INFO:src.core.models.builder:LoRA: 2,048 обучаемых из 1,034,320 (0.1980%)
INFO:src.core.models.builder:Загрузка базовой архитектуры: HuggingFaceM4/tiny-random-LlamaForCausalLM
INFO:src.core.models.builder:Применение квантизации BitsAndBytes.


[r=8, alpha=16] | End Loss: 10.3706 | Peak Mem: 1303.4 MB


Loading weights: 100%|██████████| 21/21 [00:00<00:00, 2333.75it/s]
INFO:src.core.models.builder:Активация Gradient Checkpointing (Экономия VRAM).
INFO:src.core.models.builder:Режим PEFT: Инициализация нового LoRA адаптера.
INFO:src.core.models.builder:LoRA: 4,096 обучаемых из 1,036,368 (0.3952%)
INFO:src.core.models.builder:Загрузка базовой архитектуры: HuggingFaceM4/tiny-random-LlamaForCausalLM
INFO:src.core.models.builder:Применение квантизации BitsAndBytes.


[r=16, alpha=32] | End Loss: 10.3602 | Peak Mem: 1304.4 MB


Loading weights: 100%|██████████| 21/21 [00:00<00:00, 2333.50it/s]
INFO:src.core.models.builder:Активация Gradient Checkpointing (Экономия VRAM).
INFO:src.core.models.builder:Режим PEFT: Инициализация нового LoRA адаптера.
INFO:src.core.models.builder:LoRA: 8,192 обучаемых из 1,040,464 (0.7873%)


[r=32, alpha=64] | End Loss: 10.3486 | Peak Mem: 1305.1 MB

=== Сводная таблица (Rank & Alpha) ===
| Target Modules              | r/alpha   |   Params (%) |   Peak Mem (MB) |   Time (s) |   Start Loss |   End Loss |
|:----------------------------|:----------|-------------:|----------------:|-----------:|-------------:|-----------:|
| q_proj+k_proj+v_proj+o_proj | 8/8       |       0.1980 |       1098.9000 |    24.3000 |      10.3786 |    10.3758 |
| q_proj+k_proj+v_proj+o_proj | 8/16      |       0.1980 |       1303.4000 |    25.7700 |      10.3786 |    10.3706 |
| q_proj+k_proj+v_proj+o_proj | 16/32     |       0.3952 |       1304.4000 |    26.7200 |      10.3786 |    10.3602 |
| q_proj+k_proj+v_proj+o_proj | 32/64     |       0.7873 |       1305.1000 |    28.6100 |      10.3786 |    10.3486 |

Самая мощная конфигурация: r/alpha = 32/64 с End Loss = 10.3486
